# OASIS-1 dementia ViT: lesion features (DECISION_RULES.md §5, §7)
Settings: **Internet ON**, **Accelerator: GPU T4 x2** (one GPU is used). Attach the dataset made from `oasis1_vit_input.zip`.
Run with **Save Version → Save & Run All**. Output: `features/<model>.npz` (CLS features for every condition and slice) + `validity.csv`.
No probe is fitted here; all fitting and statistics run locally with the fixed folds.

In [ ]:
!pip install -q timm==1.0.30
import os, sys, time, glob
import numpy as np, pandas as pd, torch, torch.nn.functional as F, timm

DATA = os.path.dirname(glob.glob("/kaggle/input/**/meta.csv", recursive=True)[0])
sys.path.insert(0, DATA)
from vitprobe.models import ProbedViT, REGISTRY
from vitprobe.masks import token_labels, isolate_mask, edge_keep_mask, n_cross_pairs

print("data:", DATA, "| timm", timm.__version__, "| torch", torch.__version__, "| GPUs", torch.cuda.device_count())
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "/kaggle/working/features"
os.makedirs(OUT, exist_ok=True)

images = torch.from_numpy(np.load(f"{DATA}/images.npy").astype(np.float32))   # (990, 145, 145) in [0, 1]
masks = {r: torch.from_numpy(np.load(f"{DATA}/{r}.npy").astype(np.float32))
         for r in ("mtl", "ventricles", "hippocampus", "brain")}
meta = pd.read_csv(f"{DATA}/meta.csv")
N = len(meta)
print("slices:", N)

MODELS = ["dinov2_s", "dinov2_s_reg", "dinov2_b", "dinov2_b_reg", "deit_s"]
REGIONS = ["mtl", "ventricles", "hippocampus"]
TH, TH_SENS, BRAIN_TH = 0.25, 0.5, 0.25
SEEDS = (0, 1, 2)
BATCH = 64

In [ ]:
def to_input(idx, net):
    x = images[idx][:, None]                                           # (B,1,145,145)
    x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False).clamp(0, 1)
    x = x.repeat(1, 3, 1, 1)
    mean = torch.tensor(net.mean).view(1, 3, 1, 1)
    std = torch.tensor(net.std).view(1, 3, 1, 1)
    return ((x - mean) / std).to(DEV)


def patch_frac(m, idx, patch):
    x = F.interpolate(m[idx][:, None], size=(224, 224), mode="nearest")
    return F.avg_pool2d(x, patch).flatten(1)                           # (B, n_patches)


def gens(seed, idx):
    return [torch.Generator().manual_seed(seed * 1_000_003 + int(i)) for i in idx]


def tok(patch_side, n_prefix):
    # True = the side the readout (CLS + registers) sits on
    return token_labels(patch_side.to(DEV), n_prefix, "fg")


def readout_only(region, n_prefix):
    B, n = region.shape
    T = n + n_prefix
    pref = torch.zeros(T, dtype=torch.bool, device=DEV)
    pref[:n_prefix] = True
    rt = torch.cat([torch.zeros(B, n_prefix, dtype=torch.bool), region.cpu()], 1).to(DEV)
    block = pref[None, :, None] & rt[:, None, :]
    return ~(block | block.transpose(1, 2))


def random_patches(region, brain, idx, seed):
    # same number of brain patches outside the region, chosen at random (seeded per slice)
    region, brain = region.cpu(), brain.cpu()
    sel = torch.zeros_like(region)
    for b, g in enumerate(gens(seed, idx)):
        cand = torch.nonzero(brain[b] & ~region[b]).flatten()
        k = min(int(region[b].sum()), len(cand))
        if k:
            sel[b, cand[torch.randperm(len(cand), generator=g)[:k]]] = True
    return sel


def conditions(idx, fr, brain, n_prefix, n_layers):
    # yields (name, keep mask or None, layers); the list is fixed by DECISION_RULES.md §7
    half = n_layers // 2
    early, late = set(range(half)), set(range(half, n_layers))
    yield "full", None, None
    yield "remove_brain", isolate_mask(tok(~brain, n_prefix)), None
    for ri, r in enumerate(REGIONS):
        for th in ((TH, TH_SENS) if r != "hippocampus" else (TH,)):
            sfx = "" if th == TH else "@50"
            R = fr[r] >= th
            lab = tok(~R, n_prefix)                                    # readout on the non-R side
            rm = isolate_mask(lab)
            yield f"remove_{r}{sfx}", rm, None
            for s in SEEDS:
                base = 100 * ri + (50 if sfx else 0) + s
                yield (f"random_links_{r}{sfx}#{s}",
                       edge_keep_mask(lab, None, "random", generators=gens(base, idx), n_remove=n_cross_pairs(lab)), None)
                P = random_patches(R, brain, idx, base + 10)
                yield f"random_patches_{r}{sfx}#{s}", isolate_mask(tok(~P, n_prefix)), None
            if sfx:
                continue
            yield f"only_{r}", isolate_mask(tok(R, n_prefix)), None
            yield f"remove_{r}_early", rm, early
            yield f"remove_{r}_late", rm, late
            rl0 = edge_keep_mask(lab, None, "random", generators=gens(100 * ri, idx), n_remove=n_cross_pairs(lab))
            yield f"random_links_{r}_early", rl0, early
            yield f"random_links_{r}_late", rl0, late
            yield f"readout_only_{r}", readout_only(R, n_prefix), None

In [ ]:
validity = []
for key in MODELS:
    path = f"{OUT}/{key}.npz"
    if os.path.exists(path):
        print(key, "already done")
        continue
    t0 = time.time()
    net = ProbedViT(key).to(DEV)
    patch, n_prefix, n_layers = net.patch, net.n_prefix, net.n_layers

    # validity: patched model with no mask must equal stock timm
    timm_id, _, kw = REGISTRY[key]
    stock = timm.create_model(timm_id, pretrained=True, **kw).eval().to(DEV)
    with torch.no_grad():
        x = to_input(torch.arange(32), net)
        diff = (net.features(x) - stock.forward_head(stock.forward_features(x), pre_logits=True)).abs().max().item()
    del stock

    feats = {}
    counts = {k: [] for k in ["n_mtl", "n_ventricles", "n_hippocampus", "n_mtl@50", "n_ventricles@50", "n_brain"]}
    for s in range(0, N, BATCH):
        idx = torch.arange(s, min(s + BATCH, N))
        x = to_input(idx, net)
        fr = {r: patch_frac(masks[r], idx, patch) for r in REGIONS}
        brain = patch_frac(masks["brain"], idx, patch) >= BRAIN_TH
        for r in REGIONS:
            counts[f"n_{r}"] += (fr[r] >= TH).sum(1).tolist()
        for r in ("mtl", "ventricles"):
            counts[f"n_{r}@50"] += (fr[r] >= TH_SENS).sum(1).tolist()
        counts["n_brain"] += brain.sum(1).tolist()
        for name, keep, layers in conditions(idx, fr, brain, n_prefix, n_layers):
            net.ctrl.keep, net.ctrl.layers = keep, layers
            with torch.no_grad():
                feats.setdefault(name, []).append(net.features(x).float().cpu().numpy().astype(np.float16))
        net.ctrl.keep, net.ctrl.layers = None, None
    feats = {k: np.concatenate(v) for k, v in feats.items()}
    np.savez_compressed(path, **feats, **{k: np.array(v) for k, v in counts.items()})
    validity.append(dict(model=key, patched_vs_stock_maxabs=diff, n_conditions=len(feats),
                         feat_dim=feats["full"].shape[1], seconds=round(time.time() - t0)))
    print(validity[-1], flush=True)
    del net
    torch.cuda.empty_cache()

pd.DataFrame(validity).to_csv("/kaggle/working/validity.csv", index=False)
print(pd.DataFrame(validity))